In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import GradientBoostingRegressor
import os


# Load CSV files

booking = pd.read_csv(r"C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\csv file\booking_history.csv")
revenue = pd.read_csv(r"C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\csv file\revenue.csv")
events = pd.read_csv(r"C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\csv file\events.csv")
competitor = pd.read_csv(r"C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\csv file\competitor.csv")
# Feature Engineering

month_map = {
    'January':1,'February':2,'March':3,'April':4,'May':5,'June':6,
    'July':7,'August':8,'September':9,'October':10,'November':11,'December':12
}

for df in [booking, revenue, events, competitor]:
    df["Month_Num"] = df["Month"].map(month_map)

booking["Occupancy_Factor"] = booking["Booked_Rooms"] / booking["Total_Rooms"]

df = booking.merge(revenue, on=["Month", "Room_Type", "year", "Month_Num"], how="left")
df["Revenue_per_Room"] = df["Monthly_Revenue"] / df["Booked_Rooms"]

le = LabelEncoder()
df["Room_Type_Code"] = le.fit_transform(df["Room_Type"])

event_count = events.groupby("Month_Num").size().reset_index(name="Event_Count")
comp_avg = competitor.groupby("Month_Num")["Room_price"].mean().reset_index()

df = df.merge(event_count, on="Month_Num", how="left")
df = df.merge(comp_avg, on="Month_Num", how="left")
df.fillna(0, inplace=True)


# ML Dataset

X = df[["Month_Num", "Room_Type_Code", "Revenue_per_Room", "Event_Count", "Room_price"]]
y = df["Occupancy_Factor"]


# Train ML Model

model = GradientBoostingRegressor(n_estimators=400, learning_rate=0.04, max_depth=4, random_state=42)
model.fit(X, y)


# Predict 2025 Occupancy

months = range(1, 13)
room_types = df["Room_Type_Code"].unique()

rows = []
for m in months:
    for rt in room_types:
        ev = event_count[event_count["Month_Num"] == m]["Event_Count"]
        cp = comp_avg[comp_avg["Month_Num"] == m]["Room_price"]
        rev = df[df["Room_Type_Code"] == rt]["Revenue_per_Room"].mean()
        rows.append([
            m,
            rt,
            rev if not np.isnan(rev) else 0,
            ev.values[0] if not ev.empty else 0,
            cp.values[0] if not cp.empty else 0
        ])

pred_df = pd.DataFrame(rows, columns=X.columns)
pred_df["Predicted_Occupancy"] = model.predict(pred_df)


# Apply optimistic boost

MIN_BOOST = 0.05
MAX_BOOST = 0.15
pred_df["Predicted_Occupancy"] = pred_df.apply(
    lambda r: min(r["Predicted_Occupancy"] * (1 + np.random.uniform(MIN_BOOST, MAX_BOOST)), 1.0),
    axis=1
)
pred_df["Predicted_Occupancy"] = (pred_df["Predicted_Occupancy"] * 100).round(2)


# Decode labels

month_reverse = {v: k for k, v in month_map.items()}
pred_df["Month"] = pred_df["Month_Num"].map(month_reverse)
pred_df["Room_Type"] = le.inverse_transform(pred_df["Room_Type_Code"].astype(int))

#  Save CSV

final_df = pred_df[["Room_Type", "Month", "Predicted_Occupancy"]]
final_df.rename(columns={"Predicted_Occupancy": "Predicted_Occupancy_Rate"}, inplace=True)

# Ensure folder exists
os.makedirs(r"C:\Users\SUHAIB\Hotel\model", exist_ok=True)

# Save CSV
csv_path = r"C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\model\predicted_occupancy_2025.csv"
final_df.to_csv(csv_path, index=False)

print(f"CSV created successfully at: {csv_path}")
print(final_df)


CSV created successfully at: C:\Users\SUHAIB\Downloads\Hotelupdated\Hotel\model\predicted_occupancy_2025.csv
   Room_Type      Month  Predicted_Occupancy_Rate
0   Standard    January                     90.20
1     Deluxe    January                     93.09
2     Simple    January                     67.20
3   Standard   February                     85.05
4     Deluxe   February                     82.39
5     Simple   February                     67.85
6   Standard      March                     88.85
7     Deluxe      March                     96.33
8     Simple      March                     82.09
9   Standard      April                     86.26
10    Deluxe      April                     92.33
11    Simple      April                     79.79
12  Standard        May                    100.00
13    Deluxe        May                    100.00
14    Simple        May                    100.00
15  Standard       June                    100.00
16    Deluxe       June                  

C:\Users\SUHAIB\AppData\Local\Temp\ipykernel_16308\3289412789.py:95: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.rename(columns={"Predicted_Occupancy": "Predicted_Occupancy_Rate"}, inplace=True)
